# 02 · Modeling — Next-Day PM2.5 (Option A)

Predicts the **next-day 24-h mean PM2.5** from information available through day *t*. Secondary: next-day AQI category (US EPA).

> **Run top-to-bottom only.** Requires `data/processed/dhaka_aq_master_rebuilt.parquet` from notebook 01, with the H1-2024 BLH gap already patched (`python -m src.ingest.patch_blh`).

**BLH provenance:** the 2024-01-01→06-30 `era5_boundary_layer_height` hole was a gap in the Open-Meteo ERA5 *mirror*, not the source. It is filled with **native ERA5 from the Copernicus CDS** (`src/ingest/cds_blh.py`); absent CDS credentials, the fallback is a **train-years-only (≤2023)** month×hour climatology. See `reports/data_notes.md`.

Leakage rules (enforced below): features use day *t* and earlier only; temporal split, no shuffle; no same-period satellite PM2.5 as a predictor.

In [ ]:
# === Bootstrap (local Jupyter / Antigravity IDE) ===
import os, sys, random
from pathlib import Path
import numpy as np

SEED = 42
random.seed(SEED); np.random.seed(SEED)

def _find_root(marker="config/sources.yaml"):
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / marker).exists():
            return cand
    raise FileNotFoundError(
        "Couldn't find config/sources.yaml. Open/run this notebook from inside the "
        "dhaka-pm25-forecast project folder."
    )

PROJECT_ROOT = _find_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
for d in ["data/raw","data/interim","data/processed","logs","reports/figures","models"]:
    os.makedirs(d, exist_ok=True)
print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
import yaml, re, pandas as pd
with open("config/model.yaml") as f:
    M = yaml.safe_load(f)
proc = "data/processed/dhaka_aq_master_rebuilt.parquet"
assert os.path.exists(proc), "Run notebook 01 (PROMPT_1) first to build the master."
df = pd.read_parquet(proc)
print("Loaded:", df.shape)

## Step 1 · Build the Option A target *(daily mean, joint ≥18 h — implemented)*

In [ ]:
def find_pm25_col(d):
    for pat in [r"^openaq_pm2?_?5$", r"^aq_pm2?_?5$", r"^pm2?_?5$"]:
        for c in d.columns:
            if re.match(pat, c, re.I) and not re.search(r"omaq|acag|next|lag|roll|pred", c, re.I):
                return c
    raise ValueError(f"No ground-truth PM2.5 column in {list(d.columns)}")

def find_time_col(d):
    for c in ["datetime_utc","timestamp_utc","datetime","timestamp","time_utc"]:
        if c in d.columns: return c
    for c in d.columns:
        if "date" in c.lower() or "time" in c.lower(): return c
    raise ValueError("No timestamp column")

PM25, TS = find_pm25_col(df), find_time_col(df)
MIN_HOURS = M["coverage"]["min_hourly_obs_per_day"]

def build_daily_option_a(d, pm25, ts, min_hours):
    x = d[[ts, pm25]].copy()
    x[ts] = pd.to_datetime(x[ts], utc=True, errors="coerce")
    x = x.dropna(subset=[ts])
    x["date"] = x[ts].dt.tz_convert("Asia/Dhaka").dt.normalize().dt.tz_localize(None)
    daily = (x.groupby("date")[pm25].agg(pm25_mean="mean", n_hours="count")).asfreq("D")
    daily["valid"]      = daily["n_hours"] >= min_hours
    daily["y_next"]     = daily["pm25_mean"].shift(-1)          # next-day mean = target
    daily["valid_next"] = daily["valid"].shift(-1).fillna(False)
    daily["pair_valid"] = daily["valid"] & daily["valid_next"]  # BOTH days must clear 18 h
    return daily

daily = build_daily_option_a(df, PM25, TS, MIN_HOURS)
print(f"PM2.5 col: {PM25} | daily rows: {len(daily)} | valid (t->t+1) pairs: {int(daily['pair_valid'].sum())}")

## Step 2 · Temporal split *(no shuffle — implemented)*

In [ ]:
S = M["split"]
def temporal_split(frame):
    i = frame.index
    tr = frame[i <= pd.Timestamp(S["train_end"])]
    va = frame[i.year == S["val_year"]]
    te = frame[i >= pd.Timestamp(S["test_start"])]
    return tr, va, te
print("Split config:", S)

## Step 3 · AQI categories *(US EPA 2024 PM2.5 — implemented)*

In [ ]:
import numpy as np
# US EPA 2024 24-h PM2.5 breakpoints (µg/m³). VERIFY against current EPA docs before publishing.
EPA_BINS   = [-0.1, 9.0, 35.4, 55.4, 125.4, 225.4, np.inf]
EPA_LABELS = ["Good","Moderate","USG","Unhealthy","Very Unhealthy","Hazardous"]
def to_aqi(pm): return pd.cut(pm, bins=EPA_BINS, labels=EPA_LABELS)
daily["aqi_next"] = to_aqi(daily["y_next"])
daily["aqi_next"].value_counts(dropna=True)

## Step 4 · Features — PM2.5 history *(implemented; PROMPT_2 adds met/fire/calendar)*

In [ ]:
F = M["features"]
feat = daily.copy()
feat["pm25_t"] = feat["pm25_mean"]                              # day t (legit, available)
for L in F["pm25_lags"]:
    feat[f"pm25_lag{L}"] = feat["pm25_mean"].shift(L)
for W in F["pm25_rolling_windows"]:
    feat[f"pm25_roll_mean{W}"] = feat["pm25_mean"].rolling(W).mean()
    feat[f"pm25_roll_std{W}"]  = feat["pm25_mean"].rolling(W).std()

# === PROMPT_2: day-t ERA5 aggregates + wind u/v, FIRMS lags, calendar/holiday/ramadan ===
# Leakage-safe builders live in src.features.daily (same code the CLI uses), so the
# notebook and `python -m src.run_experiment` produce an identical feature matrix.
from src.features.daily import add_exogenous_features, feature_availability

feat, FEATURES = add_exogenous_features(df, feat, M)            # merge ERA5/FIRMS/calendar by date
#   ERA5 mean/min/max + precip/radiation sums + wind_u/wind_v (circular) + BLH
#   FIRMS count/FRP at lags 0,1,2; day-level calendar. (BLH imputation flag dropped: constant 0 after CDS patch.)
#   EXCLUDED as predictors (leakage): openaq_o3, openaq_pm10 (same-station, not available
#   at forecast time) and any satellite PM2.5 — enforced inside src.features.daily.

avail = feature_availability(FEATURES)
print(f"PM2.5 history features: {[c for c in feat.columns if c.startswith('pm25_') and c!='pm25_mean']}")
print(f"Total model features: {len(FEATURES)}")
display(avail)                                                  # availability time of every column


## Step 5 · Baselines — persistence & climatology *(implemented; the bar to beat)*

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def reg_metrics(y, yhat):
    yhat = pd.Series(np.asarray(yhat), index=y.index)
    m = (~y.isna()) & (~yhat.isna())
    y, yhat = y[m], yhat[m]
    return {"rmse": float(np.sqrt(mean_squared_error(y, yhat))),
            "mae": float(mean_absolute_error(y, yhat)),
            "r2": float(r2_score(y, yhat)),
            "bias": float((yhat - y).mean()), "n": int(m.sum())}

paired = feat[feat["pair_valid"]].copy()
tr, va, te = temporal_split(paired)
print("Persistence (test):", reg_metrics(te["y_next"], te["pm25_t"].values))
doy = tr.assign(doy=tr.index.dayofyear).groupby("doy")["y_next"].mean()
print("Climatology (test):", reg_metrics(te["y_next"], te.index.dayofyear.map(doy).values))

## Step 6 · ML models — RF, XGBoost, LightGBM + tuning

**Hyperparameter selection (validation is NOT fully held out).** Hyperparameters were
selected via rolling-origin `TimeSeriesSplit(n_splits=5)` over the **combined
train+validation period (2016–2024)**; final models were **refit on training data only
(≤2023)**. The **2025 test set was untouched by any selection step**. Because the CV
folds span train+val, 2024 informed model selection — it is a tuning-aware validation
window, not a clean hold-out; the 2025 test window is the only fully untouched set.

In [ ]:
# === Step 6 · Train baselines + RF / XGBoost / LightGBM ===
# Discipline (graded harder than accuracy): scalers fit on TRAIN ONLY; XGB/LGBM
# tuned by rolling-origin TimeSeriesSplit(5) on train+val, then REFIT ON TRAIN ONLY;
# no shuffle; single seed from config. Same code path as `python -m src.run_experiment`.
from src.features.daily import model_matrix
from src.models.train import fit_scaler, apply_scaler, baselines, train_models

X, y = model_matrix(feat, FEATURES)                 # pair-valid rows, all features present
mask_tr = X.index <= pd.Timestamp(S["train_end"])
mask_va = X.index.year == S["val_year"]
mask_te = X.index >= pd.Timestamp(S["test_start"])
X_tr, y_tr = X[mask_tr], y[mask_tr]
X_va, y_va = X[mask_va], y[mask_va]
X_te, y_te = X[mask_te], y[mask_te]
X_cv = pd.concat([X_tr, X_va]).sort_index()         # train+val used ONLY for CV tuning
y_cv = pd.concat([y_tr, y_va]).sort_index()
print(f"features={len(FEATURES)} | train={len(X_tr)} val={len(X_va)} test={len(X_te)}")

scaler = fit_scaler(X_tr)                            # TRAIN ONLY
Xtr_s, Xcv_s, Xte_s = apply_scaler(scaler, X_tr), apply_scaler(scaler, X_cv), apply_scaler(scaler, X_te)

paired = feat[feat["pair_valid"]].dropna(subset=FEATURES + ["y_next"])
p_tr = paired[paired.index <= pd.Timestamp(S["train_end"])]
p_te = paired[paired.index >= pd.Timestamp(S["test_start"])]

PREDS = baselines(p_tr, p_te)                        # persistence + day-of-year climatology
MODELS, TUNING = train_models(Xtr_s, y_tr.values, Xcv_s, y_cv.values, M)
for name, mdl in MODELS.items():
    PREDS[name] = mdl.predict(Xte_s)

print("tuning:", {k: v.get("best_params") for k, v in TUNING.items()})
from src.evaluate.metrics import reg_metrics
quick = pd.DataFrame({k: reg_metrics(y_te.values, v) for k, v in PREDS.items()}).T
display(quick.sort_values("rmse").round(3))


## Step 7 · Evaluation & interpretation — *PROMPT_2 completes*

In [ ]:
# === Step 7 · Evaluation & interpretation ===
# Test-set metrics (point + 95% bootstrap CI, n_boot=1000) for ALL models + baselines,
# seasonal breakdown, residual diagnostics, AQI classification (best model), SHAP +
# permutation importance, paired significance vs persistence. Writes reports/metrics.csv,
# reports/significance_vs_persistence.csv, reports/figures/, model cards, models/ artifacts.
# Identical to `python -m src.run_experiment --config config/model.yaml`.
from src.run_experiment import evaluate_and_save
from IPython.display import Image

OUT = evaluate_and_save(MODELS, TUNING, PREDS, Xtr_s, Xte_s, y_tr, y_te, X_te, FEATURES, M)
BEST = OUT["best"]
print("Best model:", BEST)

print("\n== Test metrics (point + 95% bootstrap CI) ==")
show = ["model","rmse","rmse_ci_lo","rmse_ci_hi","mae","r2","r2_ci_lo","r2_ci_hi","bias","n","is_baseline"]
display(OUT["metrics"][show].round(3))

# Marginal CIs above cannot establish model-vs-persistence; the PAIRED tests below can.
print("\n== Significance vs persistence (paired bootstrap dMAE/dRMSE + Diebold-Mariano, h=1, HLN) ==")
print("   delta = model - persistence; delta<0 and Δ-CI excluding 0 => model beats persistence.")
display(OUT["significance"].round(4))

print("\n== By season (best model) ==");          display(OUT["seasonal"].round(3))
print("\n== AQI classification (best model) ==");  print({k: round(OUT["classification"][k],3)
                                                          for k in ("accuracy","macro_f1","n")})
print("\n== Permutation importance (top 10) ==");  display(OUT["permutation_importance"].head(10).round(4))

# Inline key figures.
for fn in [f"pred_vs_actual_{BEST}.png", f"residual_acf_{BEST}.png",
           f"shap_summary_{BEST}.png", f"confusion_matrix_{BEST}.png"]:
    display(Image(filename=f"reports/figures/{fn}"))
print("\nArtifacts: reports/metrics.csv, reports/significance_vs_persistence.csv, reports/figures/, reports/model_card_*.md, models/*")